In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
train_ds = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_ds = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
sample_sub = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv")

In [ ]:
train_ds

In [ ]:
train_ds.columns

In [ ]:
cat_data= train_ds.select_dtypes(include = "object")
num_data = train_ds.select_dtypes(include=["int", "float"]).drop(["Id", "SalePrice"],axis =1)

In [ ]:
cat_data

In [ ]:
num_data

In [ ]:
cat_cols = []
for col in cat_data.columns:
    cat_cols.append(col)


print(cat_cols)

num_cols = []
for col in num_data.columns:
    num_cols.append(col)

print(num_cols)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

In [ ]:
train_ds.columns[train_ds.isnull().any()]

In [ ]:
cat_imputer = SimpleImputer(strategy = "most_frequent").fit(train_ds[cat_cols])
num_imputer = SimpleImputer(strategy = "median").fit(train_ds[num_cols])

In [ ]:
train_ds[cat_cols] = cat_imputer.transform(train_ds[cat_cols])
train_ds[num_cols] = num_imputer.transform(train_ds[num_cols])

train_ds.columns[train_ds.isnull().any()]

In [ ]:
test_ds.columns[test_ds.isnull().any()]

In [ ]:
test_ds[cat_cols] = cat_imputer.transform(test_ds[cat_cols])
test_ds[num_cols] = num_imputer.transform(test_ds[num_cols])

test_ds.columns[test_ds.isnull().any()]

In [ ]:
train_ds.head()

In [ ]:
encoder = OneHotEncoder(sparse = False, handle_unknown = "ignore")

In [ ]:
encoder.fit(train_ds[cat_cols])

In [ ]:
train_encoded = encoder.transform(train_ds[cat_cols])
test_encoded = encoder.transform(test_ds[cat_cols])

encoded_cols = encoder.get_feature_names_out(cat_cols)
train_ds[encoded_cols] = pd.DataFrame(train_encoded, index = train_ds.index)
test_ds[encoded_cols] = pd.DataFrame(test_encoded, index=test_ds.index)

In [ ]:
train_ds

In [ ]:
encoded_cols

In [ ]:
scaler = MinMaxScaler().fit(train_ds[num_cols])

In [ ]:
train_ds[num_cols] = scaler.transform(train_ds[num_cols])
test_ds[num_cols] = scaler.transform(test_ds[num_cols])

In [ ]:
X = train_ds[num_cols + list(encoded_cols)]


In [ ]:
y  = train_ds["SalePrice"]

In [ ]:
y

In [ ]:
X

In [ ]:
import matplotlib 
import matplotlib.pyplot as plt

In [ ]:
!pip install lazypredict

In [ ]:
import lazypredict

In [ ]:
from lazypredict.Supervised import LazyRegressor

In [ ]:
from sklearn.model_selection  import train_test_split

In [ ]:
X_train, X_val , y_train, y_val = train_test_split(X, y,test_size = 0.1)

In [ ]:
len(X_train),len(y_train), len(X_test), len(y_test)

In [ ]:
from sklearn.metrics import mean_absolute_error, make_scorer

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

clf = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric= None)
models, predictions = clf.fit(X_train, X_test, y_train, y_test)


In [ ]:
print(models)

In [ ]:
test_X = test_ds[num_cols + list(encoded_cols)]

In [ ]:
from sklearn.linear_model import PoissonRegressor

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor  #ensembble
from sklearn.tree import DecisionTreeRegressor #ensembble
import xgboost
from xgboost import XGBRegressor#for stacking
from sklearn.ensemble import HistGradientBoostingRegressor   #for stacking                    
from lightgbm import LGBMRegressor #ensembble
from sklearn.ensemble import RandomForestRegressor   #for stacking                       

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin, clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
import lightgbm as lgb
from sklearn.model_selection import KFold

In [ ]:
class StackingAveragedModels(BaseEstimator, RegressorMixin, TransformerMixin):
    def __init__(self, base_models, meta_model, n_folds=5):
        self.base_models = base_models
        self.meta_model = meta_model
        self.n_folds = n_folds
   
    def fit(self, X, y):
        self.base_models_ = [list() for x in self.base_models]
        self.meta_model_ = clone(self.meta_model)
        kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=156)
        
        out_of_fold_predictions = np.zeros((X.shape[0], len(self.base_models)))
        for i, model in enumerate(self.base_models):
            for train_index, holdout_index in kfold.split(X, y):
                instance = clone(model)
                self.base_models_[i].append(instance)
                instance.fit(X[train_index], y[train_index])
                y_pred = instance.predict(X[holdout_index])
                out_of_fold_predictions[holdout_index, i] = y_pred
                
        self.meta_model_.fit(out_of_fold_predictions, y)
        return self
    def predict(self, X):
        meta_features = np.column_stack([
            np.column_stack([model.predict(X) for model in base_models]).mean(axis=1)
            for base_models in self.base_models_ ])
        return self.meta_model_.predict(meta_features)

In [ ]:
rf_model = RandomForestRegressor(max_depth =10,min_samples_split = 6, min_samples_leaf = 2)

GBoost = HistGradientBoostingRegressor(learning_rate  =0.2,max_depth =6,min_samples_leaf =2,max_iter = 1000,max_leaf_nodes =50, random_state =42)

xgb_model = XGBRegressor(booster ="gbtree", verbosity=2,eta =0.1,max_depth =8,subsample =1,)
stacked_averaged_models = StackingAveragedModels(base_models = (rf_model, GBoost),
                                                 meta_model = xgb_model)

In [ ]:
from lightgbm import LGBMRegressor

In [ ]:
stacked_averaged_models.fit(X.values,y)

In [ ]:
model_lgb = lgb.LGBMRegressor(booster ="gbtree", verbosity=2,eta =0.1,max_depth =8,subsample =2,bagging_fraction= 0.8,).fit(X,y)
etr_model = ExtraTreesRegressor(max_depth =50,min_samples_split =4,min_samples_leaf = 2,random_state =67).fit(X,y)
dt_model =DecisionTreeRegressor(max_depth =10,min_samples_split = 6, min_samples_leaf = 2).fit(X,y)

In [ ]:
lgb_pred = model_lgb.predict(test_X)
etr_pred = etr_model.predict(test_X)
dt_pred = dt_model.predict(test_X)

In [ ]:
ensemble = 0.4*etr_pred + 0.35*dt_pred + 0.25*lgb_pred

In [ ]:
stacking_preds=stacked_averaged_models.predict(test_X)

In [ ]:
final_preds= .6*ensemble + 0.4*stacking_preds

In [ ]:
final_preds

In [ ]:
sample_sub.SalePrice = final_preds

In [ ]:
sample_sub.to_csv("house1.csv",index =False)